# Template

In [10]:
import polars as pl

import src.social_groups.polars_columns as plc
from social_groups.analysis.polars_transformations import make_group_constellation
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)


In [5]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [8]:
from social_groups.analysis.definitions import defs

changed_prompt = defs.load_fn().load_asset_value("changed_prompt_mad")
original_prompt = defs.load_fn().load_asset_value("hetero_mad")

2026-02-25 14:21:41 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/changed_prompt_mad.parquet using PolarsParquetIOManager...
2026-02-25 14:21:41 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/hetero_mad.parquet using PolarsParquetIOManager...


In [19]:
(
    apply_parsing_and_group_decision(
        original_prompt.with_columns(make_group_constellation()), parser, comparer, group_reply
    )
    .group_by("group_constellation")
    .agg(
        pl.col("is_correct").mean().alias(plc.accuracy + "_original_mad")
    ).join(
        (
            apply_parsing_and_group_decision(changed_prompt, parser, comparer, group_reply)
            .group_by("group_constellation")
            .agg(
                pl.col("is_correct").mean().alias(plc.accuracy + "_changed_mad")
            )
        ),

        on="group_constellation"
    )
    .with_columns(
        pl.exclude("group_constellation", plc.accuracy + "_original_mad")
        .sub(pl.col(plc.accuracy + "_original_mad"))
        .name.prefix("delta_")
    )
)

group_constellation,accuracy_original_mad,accuracy_changed_mad,delta_accuracy_changed_mad
str,f64,f64,f64
"""HHH""",0.73,0.69,-0.04
"""LLL""",0.41,0.41,0.0
"""HHL""",0.65,0.66,0.01
"""HHM""",0.7,0.7,0.0
"""HLL""",0.47,0.48,0.01
"""LLM""",0.56,0.55,-0.01
"""LMM""",0.63,0.64,0.01
"""HMM""",0.57,0.63,0.06
"""MMM""",0.58,0.61,0.03


Probing the Chats, there is still no coherence to actually "reason" about the answers given and providing feedback.
It is more like a

Probing the chats, there is still no "reasoning about former answers", they basically answer the same question again, with more stuff in the context.

Can we somehow show that this is the case?
 -> MAD = more context, answer again?